# Severstal One-Stage New Models (Colab)

This notebook runs a lightweight, one-stage OSR comparison on Severstal using only repo-native training, `torch`, and `scikit-learn` models.

Models evaluated:
- `global_mahalanobis`
- `global_knn_k5`
- `one_class_svm_rbf`
- `isolation_forest`


In [ ]:
# 1) Runtime preflight
import sys
import numpy as np
import torch
import torchvision

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("Torch:", torch.__version__)
print("TorchVision:", torchvision.__version__)

msg = (
    "Incompatible runtime. Required stack: numpy==1.26.x, "
    "torch==2.2.2(+cu121), torchvision==0.17.2(+cu121). "
    "Please switch runtime/restart with the pinned stack before running."
)

assert np.__version__.startswith("1.26"), msg
assert torch.__version__.startswith("2.2.2"), msg
assert torchvision.__version__.startswith("0.17.2"), msg

print("Runtime preflight passed.")


In [ ]:
# 2) Repo setup (clone/pull + sys.path)
from pathlib import Path
import os
import subprocess
import sys

REPO = Path('/content/FYP-code')
if not REPO.exists():
    subprocess.check_call(['git', 'clone', 'https://github.com/spinelessknave8/FYP_code.git', str(REPO)])

subprocess.check_call(['git', '-C', str(REPO), 'fetch', 'origin'])
subprocess.check_call(['git', '-C', str(REPO), 'reset', '--hard', 'origin/main'])

os.chdir(REPO)
for p in [str(REPO), str(REPO / 'severstral-osr' / 'src')]:
    if p not in sys.path:
        sys.path.insert(0, p)

print('Repo ready at', REPO)


In [ ]:
# 3) Drive mount + dataset checks
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive', force_remount=True)

DATA_ROOT = Path('/content/drive/MyDrive/datasets/severstal')
TRAIN_CSV = DATA_ROOT / 'train.csv'
TRAIN_IMAGES = DATA_ROOT / 'train_images'
OUT_ROOT = Path('/content/drive/MyDrive/fyp_outputs/severstral_new_models')
OUT_ROOT.mkdir(parents=True, exist_ok=True)

assert TRAIN_CSV.exists(), f'Missing {TRAIN_CSV}'
assert TRAIN_IMAGES.exists() and TRAIN_IMAGES.is_dir(), f'Missing {TRAIN_IMAGES}'

print('Dataset root:', DATA_ROOT)
print('Output root :', OUT_ROOT)


In [ ]:
# 4) Build a small pilot split (normal / known / unknown)
import json
import random

from data import collect_single_label_defect_samples, collect_normal_image_paths, stratified_split

SEED = 42
rng = random.Random(SEED)

# Prefer the standard split; auto-fallback to available classes if needed.
preferred_known = ['Class_1', 'Class_2', 'Class_3']
all_single = collect_single_label_defect_samples(str(DATA_ROOT), 'train.csv', 'train_images')
all_classes = sorted({cls for _, cls in all_single})
known_classes = [c for c in preferred_known if c in all_classes]
if len(known_classes) < 3:
    known_classes = all_classes[:3]
unknown_classes = [c for c in all_classes if c not in known_classes]

if not known_classes or not unknown_classes:
    raise RuntimeError(f'Cannot build known/unknown split from classes: {all_classes}')

known_all = [(p, c) for p, c in all_single if c in known_classes]
unknown_all = [(p, c) for p, c in all_single if c in unknown_classes]
normal_all = collect_normal_image_paths(str(DATA_ROOT), 'train.csv', 'train_images')
rng.shuffle(normal_all)

known_train, known_val, known_test = stratified_split(known_all, train_ratio=0.7, val_ratio=0.15, seed=SEED)

# Small pilot caps to keep runtime practical in Colab.
MAX_NORMAL_TRAIN = 1000
MAX_NORMAL_VAL = 300
MAX_NORMAL_TEST = 300
MAX_KNOWN_TRAIN = 900
MAX_KNOWN_VAL = 300
MAX_KNOWN_TEST = 300
MAX_UNKNOWN_TEST = 300

normal_train = normal_all[:MAX_NORMAL_TRAIN]
normal_val = normal_all[MAX_NORMAL_TRAIN:MAX_NORMAL_TRAIN + MAX_NORMAL_VAL]
normal_test = normal_all[MAX_NORMAL_TRAIN + MAX_NORMAL_VAL:MAX_NORMAL_TRAIN + MAX_NORMAL_VAL + MAX_NORMAL_TEST]

rng.shuffle(known_train)
rng.shuffle(known_val)
rng.shuffle(known_test)
rng.shuffle(unknown_all)

known_train = known_train[:MAX_KNOWN_TRAIN]
known_val = known_val[:MAX_KNOWN_VAL]
known_test = known_test[:MAX_KNOWN_TEST]
unknown_test = unknown_all[:MAX_UNKNOWN_TEST]

if min(len(normal_train), len(normal_test), len(known_train), len(known_test), len(unknown_test)) == 0:
    raise RuntimeError('Pilot split produced empty groups; adjust caps or class selection.')

pilot = {
    'seed': SEED,
    'known_classes': known_classes,
    'unknown_classes': unknown_classes,
    'normal_train': normal_train,
    'normal_val': normal_val,
    'normal_test': normal_test,
    'known_train': known_train,
    'known_val': known_val,
    'known_test': known_test,
    'unknown_test': unknown_test,
}

manifest_path = OUT_ROOT / 'pilot_manifest.json'
manifest_path.write_text(json.dumps(pilot, indent=2))

print('Known classes  :', known_classes)
print('Unknown classes:', unknown_classes)
print('Counts:', {
    'normal_train': len(normal_train),
    'normal_val': len(normal_val),
    'normal_test': len(normal_test),
    'known_train': len(known_train),
    'known_val': len(known_val),
    'known_test': len(known_test),
    'unknown_test': len(unknown_test),
})
print('Saved:', manifest_path)


In [ ]:
# 5) Train classifier on known classes (existing ResNet50 code path)
import torch
from pathlib import Path

from train_classifier import main as train_classifier_main

SPLIT_NAME = 'split_colab_new_models'
CFG_PATH = Path('/content/FYP-code/severstral-osr/configs') / f'{SPLIT_NAME}.colab.yaml'
PIPE_OUT = OUT_ROOT / 'pipeline_outputs'
PIPE_OUT.mkdir(parents=True, exist_ok=True)

device_token = '"cuda"' if torch.cuda.is_available() else '"cpu"'
cfg_lines = [
    f"known_classes: {pilot['known_classes']}",
    f"device: {device_token}",
    'output_dir: "' + PIPE_OUT.as_posix() + '"',
    'batch_size: 32',
    'num_workers: 2',
    'resnet:',
    '  epochs: 4',
    '  lr: 1e-3',
    '  weight_decay: 1e-4',
    '  pretrained: true',
    'severstal:',
    '  data_root: "' + DATA_ROOT.as_posix() + '"',
    '  train_csv: "train.csv"',
    '  images_dir: "train_images"',
]
CFG_PATH.write_text("\n".join(cfg_lines) + "\n")

print('Config written:', CFG_PATH)
train_classifier_main(str(CFG_PATH))
print('Classifier training complete.')


In [ ]:
# 6) Extract embeddings (normal / known / unknown groups)
import json
import numpy as np
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from src.models.resnet50 import build_resnet50
from src.models.embedding import EmbeddingExtractor

manifest = json.loads((OUT_ROOT / 'pilot_manifest.json').read_text())
split_dir = PIPE_OUT / SPLIT_NAME
clf_ckpt = split_dir / 'classifier' / 'classifier.pt'
assert clf_ckpt.exists(), f'Missing classifier checkpoint: {clf_ckpt}'

ckpt = torch.load(clf_ckpt, map_location='cpu')
class_to_idx = ckpt['class_to_idx']

model = build_resnet50(num_classes=len(class_to_idx), pretrained=False)
model.load_state_dict(ckpt['model_state'])
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device).eval()
extractor = EmbeddingExtractor(model).to(device).eval()

tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

class PathDataset(Dataset):
    def __init__(self, rows, include_labels=False):
        self.rows = rows
        self.include_labels = include_labels
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, idx):
        row = self.rows[idx]
        if self.include_labels:
            path, cls = row
            y = class_to_idx[cls]
        else:
            path = row
            y = -1
        x = tf(Image.open(path).convert('RGB'))
        return x, y

def embed_rows(rows, include_labels=False, bs=32):
    ds = PathDataset(rows, include_labels=include_labels)
    dl = DataLoader(ds, batch_size=bs, shuffle=False, num_workers=2)
    embs, ys = [], []
    with torch.no_grad():
        for x, y in dl:
            x = x.to(device)
            z = extractor(x)
            embs.append(z.cpu().numpy())
            ys.append(y.numpy())
    return np.concatenate(embs, axis=0), np.concatenate(ys, axis=0)

emb_dir = OUT_ROOT / 'embeddings'
emb_dir.mkdir(parents=True, exist_ok=True)

group_defs = [
    ('normal_train', manifest['normal_train'], False),
    ('normal_val', manifest['normal_val'], False),
    ('normal_test', manifest['normal_test'], False),
    ('known_train', manifest['known_train'], True),
    ('known_val', manifest['known_val'], True),
    ('known_test', manifest['known_test'], True),
    ('unknown_test', manifest['unknown_test'], False),
]

for name, rows, has_labels in group_defs:
    emb, labels = embed_rows(rows, include_labels=has_labels)
    np.savez(emb_dir / f'{name}.npz', embeddings=emb, labels=labels)
    print(f'{name}:', emb.shape)

extractor.close()
print('Embeddings saved to', emb_dir)


In [ ]:
# 7) Run one-stage models only + 8) Evaluate at fixed FPR_normal operating points
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.neighbors import NearestNeighbors
from sklearn.svm import OneClassSVM
from sklearn.ensemble import IsolationForest


def load_emb(name):
    obj = np.load(OUT_ROOT / 'embeddings' / f'{name}.npz')
    return obj['embeddings']

E_normal_train = load_emb('normal_train')
E_normal_val = load_emb('normal_val')
E_normal_test = load_emb('normal_test')
E_known_test = load_emb('known_test')
E_unknown_test = load_emb('unknown_test')

fpr_targets = [0.05, 0.10, 0.20]


def mahalanobis_fit(E):
    mu = E.mean(axis=0)
    cov = np.cov(E, rowvar=False) + 1e-3 * np.eye(E.shape[1])
    inv = np.linalg.inv(cov)
    return mu, inv


def mahalanobis_score(E, mu, inv):
    d = E - mu
    return np.einsum('bi,ij,bj->b', d, inv, d)


def evaluate_scores(model_name, s_normal_calib, s_normal_eval, s_known, s_unknown):
    s_defect = np.concatenate([s_known, s_unknown])
    y = np.concatenate([np.zeros(len(s_normal_eval)), np.ones(len(s_defect))])
    s = np.concatenate([s_normal_eval, s_defect])
    auroc = float(roc_auc_score(y, s))

    rows = []
    for fpr in fpr_targets:
        tau = float(np.quantile(s_normal_calib, 1.0 - fpr))
        rows.append({
            'model': model_name,
            'fpr_target': fpr,
            'threshold': tau,
            'AUROC_defect_screening': auroc,
            'TPR_defect@FPR': float(np.mean(s_defect > tau)),
            'TPR_unknown@FPR': float(np.mean(s_unknown > tau)),
            'FPR_known_as_def@FPR': float(np.mean(s_known > tau)),
            'FPR_normal_realized': float(np.mean(s_normal_eval > tau)),
        })
    return rows

rows = []

# global_mahalanobis
mu, inv = mahalanobis_fit(E_normal_train)
rows += evaluate_scores(
    'global_mahalanobis',
    mahalanobis_score(E_normal_val, mu, inv),
    mahalanobis_score(E_normal_test, mu, inv),
    mahalanobis_score(E_known_test, mu, inv),
    mahalanobis_score(E_unknown_test, mu, inv),
)

# global_knn_k5
knn = NearestNeighbors(n_neighbors=5, metric='euclidean').fit(E_normal_train)
score_knn = lambda E: knn.kneighbors(E, return_distance=True)[0].mean(axis=1)
rows += evaluate_scores(
    'global_knn_k5',
    score_knn(E_normal_val),
    score_knn(E_normal_test),
    score_knn(E_known_test),
    score_knn(E_unknown_test),
)

# one_class_svm_rbf
ocsvm = OneClassSVM(kernel='rbf', gamma='scale', nu=0.05)
ocsvm.fit(E_normal_train)
score_ocsvm = lambda E: -ocsvm.score_samples(E)
rows += evaluate_scores(
    'one_class_svm_rbf',
    score_ocsvm(E_normal_val),
    score_ocsvm(E_normal_test),
    score_ocsvm(E_known_test),
    score_ocsvm(E_unknown_test),
)

# isolation_forest
iforest = IsolationForest(n_estimators=200, contamination=0.05, random_state=42)
iforest.fit(E_normal_train)
score_if = lambda E: -iforest.score_samples(E)
rows += evaluate_scores(
    'isolation_forest',
    score_if(E_normal_val),
    score_if(E_normal_test),
    score_if(E_known_test),
    score_if(E_unknown_test),
)

summary = pd.DataFrame(rows)
summary_path = OUT_ROOT / 'one_stage_summary.csv'
summary.to_csv(summary_path, index=False)

print('Saved:', summary_path)
display(summary)


In [ ]:
# 9) Write simple plots to Drive
import matplotlib.pyplot as plt
import pandas as pd

summary = pd.read_csv(OUT_ROOT / 'one_stage_summary.csv')

# Plot 1: AUROC per model
auroc_df = summary[['model', 'AUROC_defect_screening']].drop_duplicates().sort_values('AUROC_defect_screening', ascending=False)
plt.figure(figsize=(8, 4))
plt.bar(auroc_df['model'], auroc_df['AUROC_defect_screening'])
plt.ylim(0.0, 1.0)
plt.xticks(rotation=20, ha='right')
plt.title('Defect Screening AUROC')
plt.tight_layout()
auroc_png = OUT_ROOT / 'plot_auroc_defect_screening.png'
plt.savefig(auroc_png, dpi=140)
plt.show()

# Plot 2: TPR_unknown at fixed FPR targets
pivot = summary.pivot(index='model', columns='fpr_target', values='TPR_unknown@FPR')
pivot = pivot.sort_index()
plt.figure(figsize=(8, 4))
for fpr in sorted(pivot.columns):
    plt.plot(pivot.index, pivot[fpr], marker='o', label=f'FPR={int(fpr*100)}%')
plt.ylim(0.0, 1.0)
plt.xticks(rotation=20, ha='right')
plt.title('TPR_unknown at Fixed FPR_normal')
plt.legend()
plt.tight_layout()
unknown_png = OUT_ROOT / 'plot_tpr_unknown_vs_fpr.png'
plt.savefig(unknown_png, dpi=140)
plt.show()

print('Saved plots:')
print('-', auroc_png)
print('-', unknown_png)


## 10) Rerun After Disconnect

1. Open this notebook and run all cells from top to bottom.
2. If runtime versions changed, fix the environment first so preflight passes (`numpy 1.26.x`, `torch 2.2.2`, `torchvision 0.17.2`).
3. Existing files in `/content/drive/MyDrive/fyp_outputs/severstral_new_models/` will be reused/overwritten.
